## 高级约束设计
1. 使用函数约束或者触发器来保证管理者的工资必须高于他所管理的任何一个员工。
2. 使用触发器保证任何一个员工工资的变化额度，都应该体现在他所在部门的预算上面，本质上这相当于实现了一个物化视图的一致性维护机制（触发器应该考虑到员工改变工作部门的情况，从一致性维护效率的角度，完全重算当然最简单，但希望还是实现基于更新行的增量更新）。

In [1]:
import sqlite3

In [2]:
# 连接到数据库
conn = sqlite3.connect('employee_dept.db')
cursor = conn.cursor()

### 创建经理工资检查触发器
创建经理工资检查触发器：更新经理时检查，插入/更新员工时检查。
- `check_manager_salary_update`.
- `check_employee_salary_insert`.
- `check_employee_salary_update`.

之所以我们这样来创建触发器，是因为SQLite3不支持一些比较高级的Trigger特性。

注意：不可能插入经理，只可能修改经理。

In [3]:
cursor.execute('''
DROP TRIGGER IF EXISTS check_manager_salary_update
''')
conn.commit()
print("已经删除Trigger: check_manager_salary_update")

已经删除Trigger: check_manager_salary_update


In [4]:
cursor.execute('''
-- 更新时检查经理工资
CREATE TRIGGER check_manager_salary_update
BEFORE UPDATE OF salary, dno, eno ON Emp
FOR EACH ROW
WHEN NEW.eno = (SELECT manager FROM Dept WHERE dno = NEW.dno)
  AND (SELECT IFNULL(MAX(salary), 0)
       FROM Emp
       WHERE dno = NEW.dno
         AND eno != NEW.eno) >= NEW.salary
BEGIN
  SELECT RAISE(ABORT, '经理的工资必须高于所有员工');
END;
''')
conn.commit()
print("成功创建更新经理时经理工资检查触发器")

成功创建更新经理时经理工资检查触发器


In [5]:
cursor.execute('''
DROP TRIGGER IF EXISTS check_employee_salary_insert
''')
conn.commit()
print("已经删除Trigger: check_employee_salary_insert")

已经删除Trigger: check_employee_salary_insert


In [6]:
cursor.execute('''
-- 插入时检查员工工资
CREATE TRIGGER check_employee_salary_insert
BEFORE INSERT ON Emp
FOR EACH ROW
WHEN NEW.eno != (SELECT manager FROM Dept WHERE dno = NEW.dno)
  AND NEW.salary >= (
        SELECT salary
        FROM Emp
        WHERE eno = (SELECT manager FROM Dept WHERE dno = NEW.dno)
      )
BEGIN
  SELECT RAISE(ABORT, '员工工资不能高于经理');
END;
''')
conn.commit()
print("成功创建插入员工时经理工资检查触发器")

成功创建插入员工时经理工资检查触发器


In [7]:
cursor.execute('''
DROP TRIGGER IF EXISTS check_employee_salary_update
''')
conn.commit()
print("已经删除Trigger: check_employee_salary_update")

已经删除Trigger: check_employee_salary_update


In [8]:
cursor.execute('''
-- 更新时检查员工工资
CREATE TRIGGER check_employee_salary_update
BEFORE UPDATE OF salary, dno, eno ON Emp
FOR EACH ROW
WHEN NEW.eno != (SELECT manager FROM Dept WHERE dno = NEW.dno)
  AND NEW.salary >= (
        SELECT salary
        FROM Emp
        WHERE eno = (SELECT manager FROM Dept WHERE dno = NEW.dno)
      )
BEGIN
  SELECT RAISE(ABORT, '员工工资不能高于经理');
END;
''')
conn.commit()
print("成功创建更新员工时经理工资检查触发器")

成功创建更新员工时经理工资检查触发器


### 创建预算维护触发器
同样的，分别对于`INSERT`、`DELETE`、`UPDATE`创建触发器。但是考虑到`UPDATE`的时候部门变动的情况，我们需要创建两个触发器。
- `update_budget_on_insert`.
- `update_budget_on_delete`.
- `update_budget_on_dept_change`.
- `update_budget_on_salary_change`.

In [9]:
cursor.execute('''
DROP TRIGGER IF EXISTS update_budget_on_insert
''')
conn.commit()
print("已经删除Trigger: update_budget_on_insert")

已经删除Trigger: update_budget_on_insert


In [10]:
# 创建预算维护触发器 - INSERT
cursor.execute('''
CREATE TRIGGER update_budget_on_insert
AFTER INSERT ON Emp
FOR EACH ROW
BEGIN
    UPDATE Dept SET budget = budget + NEW.salary WHERE dno = NEW.dno;
END;
''')
conn.commit()
print("成功创建插入员工时预算维护触发器")

成功创建插入员工时预算维护触发器


In [11]:
cursor.execute('''
DROP TRIGGER IF EXISTS update_budget_on_delete
''')
conn.commit()
print("已经删除Trigger: update_budget_on_delete")

已经删除Trigger: update_budget_on_delete


In [12]:
# 创建预算维护触发器 - DELETE
cursor.execute('''
CREATE TRIGGER update_budget_on_delete
AFTER DELETE ON Emp
FOR EACH ROW
BEGIN
    UPDATE Dept SET budget = budget - OLD.salary WHERE dno = OLD.dno;
END;
''')
conn.commit()
print("成功创建删除员工时预算维护触发器")

成功创建删除员工时预算维护触发器


In [13]:
cursor.execute('''
DROP TRIGGER IF EXISTS update_budget_on_dept_change
''')
conn.commit()
print("已经删除Trigger: update_budget_on_dept_change")

已经删除Trigger: update_budget_on_dept_change


In [14]:
# 创建预算维护触发器 - UPDATE when dept changes.
cursor.execute('''
-- 部门变更时调整旧/新部门预算
CREATE TRIGGER update_budget_on_dept_change
AFTER UPDATE OF dno, salary ON Emp
FOR EACH ROW
WHEN NEW.dno <> OLD.dno
BEGIN
  UPDATE Dept SET budget = budget - OLD.salary WHERE dno = OLD.dno;
  UPDATE Dept SET budget = budget + NEW.salary WHERE dno = NEW.dno;
END;
''')
conn.commit()
print("成功创建更新员工为不同部门时预算维护触发器")

成功创建更新员工为不同部门时预算维护触发器


In [15]:
cursor.execute('''
DROP TRIGGER IF EXISTS update_budget_on_salary_change
''')
conn.commit()
print("已经删除Trigger: update_budget_on_salary_change")

已经删除Trigger: update_budget_on_salary_change


In [16]:
# 创建预算维护触发器 - UPDATE when dept NOT changes.
cursor.execute('''
-- 同部门工资变动时按差值调整
CREATE TRIGGER update_budget_on_salary_change
AFTER UPDATE OF salary ON Emp
FOR EACH ROW
WHEN NEW.dno = OLD.dno
BEGIN
  UPDATE Dept
    SET budget = budget + (NEW.salary - OLD.salary)
    WHERE dno = NEW.dno;
END;
''')
conn.commit()
print("成功创建更新员工为相同部门时预算维护触发器")

成功创建更新员工为相同部门时预算维护触发器


下面我们对于触发器进行测试。

In [17]:
print("1. 更新经理薪资测试（应失败）：")
try:
    cursor.execute("UPDATE Emp SET salary = 1000 WHERE eno = '0003';")
    conn.commit()
except sqlite3.DatabaseError as e:
    print("Error:", e)

1. 更新经理薪资测试（应失败）：
Error: 经理的工资必须高于所有员工


In [18]:
print("1. 更新经理薪资测试（应成功）：")
try:
    # 提薪至25000
    cursor.execute("UPDATE Emp SET salary = 25000.0 WHERE eno = '0003'")
    conn.commit()
    print("更新经理薪资成功")
except sqlite3.DatabaseError as e:
    print("Error:", e)

1. 更新经理薪资测试（应成功）：
更新经理薪资成功


In [19]:
print("2. 插入员工薪资测试（应失败）：")
try:
    cursor.execute("INSERT INTO Emp VALUES ('0005','新人','2000-02-02',2,'秘书',16000.0,'0001');")
    conn.commit()
except sqlite3.DatabaseError as e:
    print("Error:", e)

2. 插入员工薪资测试（应失败）：
Error: 员工工资不能高于经理


In [20]:
print("2. 插入员工薪资测试（应成功）：")
try:
    # 插入薪资14000，低于经理
    cursor.execute("INSERT INTO Emp VALUES ('0006','新教职','2001-12-31',3,'秘书',14000.0,'0001')")
    conn.commit()
    print("成功插入员工")
except sqlite3.DatabaseError as e:
    print("Error:", e)

2. 插入员工薪资测试（应成功）：
成功插入员工


In [21]:
print("3. 更新员工薪资测试（应失败）：")
try:
    cursor.execute("UPDATE Emp SET salary = 30000 WHERE eno = '0004';")
    conn.commit()
except sqlite3.DatabaseError as e:
    print("Error:", e)

3. 更新员工薪资测试（应失败）：
Error: 员工工资不能高于经理


In [ ]:
print("3. 更新员工薪资测试（应成功）：")
try:
    # 调薪到6500，低于经理
    cursor.execute("UPDATE Emp SET salary = 6500.0 WHERE eno = '0004'")
    conn.commit()
    print("更新员工薪资成功")
except sqlite3.DatabaseError as e:
    print("Error:", e)

3. 更新员工薪资测试（应成功）：
更新员工薪资成功


In [23]:
def show_budget(dno):
    # 使用 ? 占位符，并通过第二个参数传入参数元组
    budgets = cursor.execute(
        "SELECT * FROM Dept WHERE dno = ?",
        (dno,)
    ).fetchall()
    print(f"Dept budgets for {dno}:", budgets)

In [ ]:
print("4. 预算 插入触发测试（部门0002预算+6000）：")
show_budget('0002')
try:
    cursor.execute("INSERT INTO Emp VALUES ('0008','测试','1999-10-01',2,'教师',6000.0,'0002')")
    conn.commit()
    show_budget('0002')
except sqlite3.DatabaserError as e:
    print("Error:", e)

4. 预算插入触发测试（部门0002预算+6000）：
Dept budgets for 0002: [('0002', '数学学院', 800000.0, '0002')]
Dept budgets for 0002: [('0002', '数学学院', 806000.0, '0002')]


In [ ]:
print("5. 预算 删除触发测试（部门0002预算-6000）：")
show_budget('0002')
try:
    cursor.execute("DELETE FROM Emp WHERE eno = '0008'")
    conn.commit()
    show_budget('0002')
except sqlite3.DatabaseError as e:
    print("Error:", e)

5. 预算删除触发测试（部门0002预算-6000）：
Dept budgets for 0002: [('0002', '数学学院', 806000.0, '0002')]
Dept budgets for 0002: [('0002', '数学学院', 800000.0, '0002')]


In [ ]:
print("6. 预算 部门变更触发测试（0004从部门0003更换至部门0002）")
show_budget('0002')
show_budget('0003')
try:
    cursor.execute("UPDATE Emp SET dno = '0002' WHERE eno = '0004'")
    conn.commit()
    show_budget('0002')
    show_budget('0003')
except sqlite3.DatabaseError as e:
    print("Error:", e)

6. 预算部门变更触发测试（0004从部门0003更换至部门0002）
Dept budgets for 0002: [('0002', '数学学院', 800000.0, '0002')]
Dept budgets for 0003: [('0003', '智能学院', 1204500.0, '0003')]
Dept budgets for 0002: [('0002', '数学学院', 806500.0, '0002')]
Dept budgets for 0003: [('0003', '智能学院', 1198000.0, '0003')]


In [30]:
print("7. 预算 薪资修改出发测试（0004修改薪资为3500）")
show_budget('0002')
try:
    cursor.execute("UPDATE Emp SET salary = 3500 WHERE eno = '0004'")
    conn.commit()
    show_budget('0002')
except sqlite3.DatabaseError as e:
    print("Error", e)

7. 预算 薪资修改出发测试（0004修改薪资为3500）
Dept budgets for 0002: [('0002', '数学学院', 806500.0, '0002')]
Dept budgets for 0002: [('0002', '数学学院', 803500.0, '0002')]
